In [8]:
import pandas as pd


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
df = pd.read_csv("netflix_titles.csv")

In [13]:
print(df.shape)
df.head()


(8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [14]:
df = df.drop_duplicates(subset="title").reset_index(drop=True)

for col in ["director", "country", "listed_in", "rating"]:
    df[col] = df[col].fillna("Unknown")

df["content_soup"] = (
    (df["listed_in"] + " ") * 2
    + df["director"] + " "
    + df["country"] + " "
    + df["rating"] + " "
    + df["type"]
)

df[["title", "content_soup"]].head()


,title,content_soup
0,Dick Johnson Is Dead,Documentaries Documentaries Kirsten Johnson Un...
1,Ganglands,"Crime TV Shows, International TV Shows, TV Act..."
2,Midnight Mass,"TV Dramas, TV Horror, TV Mysteries TV Dramas, ..."
3,Confessions of an Invisible Girl,"Children & Family Movies, Comedies Children & ..."
4,Sankofa,"Dramas, Independent Movies, International Movi..."


In [15]:
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["content_soup"])
print("TF-IDF matrix shape:", tfidf_matrix.shape)



TF-IDF matrix shape: (8787, 6569)


In [16]:
sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Similarity matrix shape:", sim_matrix.shape)


Similarity matrix shape: (8787, 8787)


In [17]:
def recommend(title, df, sim_matrix, top_n=5):
    matches = df.index[df["title"].str.lower() == title.lower()]
    if len(matches) == 0:
        close = df[df["title"].str.contains(title, case=False, na=False)]
        if close.empty:
            raise ValueError(f"Title '{title}' not found in dataset.")
        idx = close.index[0]
    else:
        idx = matches[0]

    scores = list(enumerate(sim_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != idx][:top_n]

    result = df.iloc[[i for i, _ in scores]][["title", "type", "listed_in", "release_year"]].copy()
    result["similarity_score"] = [round(s, 3) for _, s in scores]
    return result.reset_index(drop=True)

recommend("Ganglands", df, sim_matrix, top_n=5)


,title,type,listed_in,release_year,similarity_score
0,Lupin,TV Show,"Crime TV Shows, International TV Shows, TV Act...",2021,0.853
1,Crime Time,TV Show,"Crime TV Shows, International TV Shows, TV Act...",2017,0.853
2,Fatal Destiny,TV Show,"Crime TV Shows, International TV Shows, TV Act...",2016,0.810
3,Agent Raghav,TV Show,"Crime TV Shows, International TV Shows, TV Act...",2015,0.806
4,Smoking,TV Show,"Crime TV Shows, International TV Shows, TV Act...",2018,0.806


In [18]:
def evaluate_genre_overlap(title, df, sim_matrix, top_n=5):
    idx = df.index[df["title"].str.lower() == title.lower()][0]
    source_genres = set(g.strip() for g in df.loc[idx, "listed_in"].split(","))

    recs = recommend(title, df, sim_matrix, top_n=top_n)
    overlap_count = 0
    for rec_title in recs["title"]:
        rec_idx = df.index[df["title"] == rec_title][0]
        rec_genres = set(g.strip() for g in df.loc[rec_idx, "listed_in"].split(","))
        if source_genres & rec_genres:
            overlap_count += 1

    return overlap_count / top_n

quality = evaluate_genre_overlap("Ganglands", df, sim_matrix, top_n=5)
print(f"Genre-overlap quality score: {quality:.2f}")


Genre-overlap quality score: 1.00


In [ ]:
# ============================================================
# CELL 8 — Interactive demo (proof it works on any title)
# ============================================================
user_title = input("Enter a Netflix title to get recommendations: ")

try:
    print(f"\nBecause you liked '{user_title}', you might also like:\n")
    recs = recommend(user_title, df, sim_matrix, top_n=5)
    print(recs.to_string(index=False))

    quality = evaluate_genre_overlap(user_title, df, sim_matrix, top_n=5)
    print(f"\nGenre-overlap quality score: {quality:.2f}")
except ValueError as e:
    print(e)